# 快速入门

点击顶部栏的 `🚀` -> `Binder` 在线运行此示例！

本示例将介绍：

* 如何通过节点连接到区块链
* 如何创建和使用账户
* 如何与合约交互或发送交易


## 连接到节点

我们需要通过节点连接到 Conflux 网络。您可以按照[本教程](https://developer.confluxnetwork.org/conflux-doc/docs/get_started/)在本地运行一个节点。在本示例中，我们使用 [Conflux 公共 RPC 端点](https://developer.confluxnetwork.org/sdks-and-tools/en/conflux_rpcs)连接到 Conflux 测试网。

In [1]:
from conflux_web3 import Web3

w3 = Web3(Web3.HTTPProvider("https://test.confluxrpc.com"))
assert w3.is_connected()

### 创建账户

在 web3 中，拥有一个账户意味着知道一个**秘密**，这被称为**私钥/密钥**。换句话说，谁知道私钥谁就拥有这个账户。因此密钥必须保密。

我们可以通过 `w3.account.create()` 创建一个新账户。这个函数会从随机生成的密钥创建一个账户。

也支持使用 `w3.account.from_key("0x....")` 来使用您的密钥（**但不要在不安全的环境中运行它！**）

In [2]:
acct = w3.account.create()
print(f"account address: {acct.address}")
# WARNING: Don't run the following line in unsafe environment, private key should be kept secret
print(f"account secret key: {acct.key.hex()}")
balance = w3.cfx.get_balance(acct.address)
# this is a random accoun, so its balance should be 0
assert balance == 0
print(f"balance for {acct.address}: {balance}")

account address: cfxtest:aaswj28188e35rh1vguksgnuz2xy4f8apye3745zxb
account secret key: 0x54d957b2485980fc2119ccce6480ad0d53219161595424fbf7dc452d59c6bc82
balance for cfxtest:aaswj28188e35rh1vguksgnuz2xy4f8apye3745zxb: 0 Drip


## 通过测试网水龙头合约获取 CFX

由于账户余额为 `0`，无法支付发送交易所需的手续费（gas）。不过，Conflux 的[赞助机制](https://forum.conflux.fun/t/conflux-sponsorship-mechanism/12764)允许用户在不支付 gas 的情况下与智能合约交互，因此我们可以从[测试网水龙头合约](https://testnet.confluxscan.net/address/cfxtest:acejjfa80vj06j2jgtz9pngkv423fhkuxj786kjr61)获取 CFX。

> 智能合约：智能合约是部署在区块链上的程序。它提供接口来完成特定的逻辑。在本例中，调用水龙头合约的 `claimCfx` 方法将给您 1000 个测试网 CFX。

In [3]:
# Firstly we will set `w3.cfx.default_account` to `acct`, 
# after that transactions can be automatically singed and sent.
w3.cfx.default_account = acct

# interact with testnet Faucet contract
faucet = w3.cfx.contract(name="Faucet")
tx_hash = faucet.functions.claimCfx().transact()

print(f"tx hash is: {tx_hash.hex()}\n"
      f"confluxscan link: https://testnet.confluxscan.net/transaction/{tx_hash.hex()}")

tx hash is: 0xf174cf4be19b33da58504ef850bfd055bd031324135093276abbee1c34df854d
conflux scan link: https://testnet.confluxscan.net/transaction/0xf174cf4be19b33da58504ef850bfd055bd031324135093276abbee1c34df854d


In [4]:
# in Conflux, the transaction will be executed only after it appears on the chain for 5 epoch
# `tx_hash.executed()` is equivalent to `w3.cfx.wait_for_transaction_receipt(tx_hash)`
tx_hash.executed()
# Drip and CFX are token units in Conflux network
# 1 CFX = 10**18 Drip
print(f"balance for {acct.address}: {w3.cfx.get_balance(acct.address).to('CFX')}")

balance for cfxtest:aaswj28188e35rh1vguksgnuz2xy4f8apye3745zxb: 1000 CFX


## 转账 CFX

现在我们有足够的余额支付交易手续费了。例如，我们可以发送 1 CFX 到零地址。

In [5]:
# Now acct has CFX
# send 1 CFX to zero address
w3.cfx.send_transaction({
    "to": w3.address.zero_address(),
    "value": 10**18,
}).executed()
print(f"balance for {acct.address}: {w3.cfx.get_balance(acct.address).to('CFX')}")

balance for cfxtest:aaswj28188e35rh1vguksgnuz2xy4f8apye3745zxb: 998.999979 CFX
